# Tips for task 3a) in worksheet 02

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tip 1</h2>
    <details>
    <summary>Click here!</summary>
    
  In case you want to use batching, ensure that your qnode is batch-aware, e.g.

  ```python
  for i in range(n_qubits):
    qml.RY(inputs[..., i], wires=i)
  ```
</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tip 2</h2>
    <details>
    <summary>Click here!</summary>
    
  The result of your qnode also needs to be reshaped to support proper batchint. So take care about the reshaping in your hybrid model `forward` definition:

  ```python
  qout = qout.unsqueeze(-1)
  ```
</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tip 3</h2>
    <details>
    <summary>Click here!</summary>
    
  You can use everything from torch for your hybrid setup, e.g. learning rate decay.
</details>
</div>

<div class="alert alert-success">
  <h2><i class="fas fa-check" style="font-size:36px"></i> &nbsp; Exemplary Solution </h2>
    <details>
    <summary>Click here!</summary>

  ```python
  # Step 1: Define the quantum circuit as a QNode
  @qml.qnode(dev, interface='torch')
  def quantum_circuit(inputs, weights):
    """
    Quantum circuit for classification.
    """
    for i in range(n_qubits):
        qml.RY(inputs[..., i]*np.pi, wires=i)

    n_layers = weights.shape[0]
    for layer in range(n_layers):
        for i in range(n_qubits):
            qml.RZ(weights[layer, i, 0], wires=i)
            qml.RY(weights[layer, i, 1], wires=i)
            qml.RZ(weights[layer, i, 2], wires=i)
        # simple entangling layer
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[n_qubits - 1, 0])

    return qml.expval(qml.PauliZ(0))

  weight_shapes = {"weights": (n_layers, n_qubits, 3)}
  qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

  class HybridQNN(nn.Module):
    def __init__(self, n_features=n_qubits):
        super().__init__()
        # Classical preprocessing layers
        self.pre_net = nn.Sequential(nn.Linear(n_features, n_qubits), nn.ReLU())
        # Quantum layer
        self.qlayer = qlayer
        # Simple post-processing layer
        self.post = nn.Linear(1, 1)
    
    
    def forward(self, x):
        # x: tensor of shape (batch, n_features)
        x = self.pre_net(x)
        qout = self.qlayer(x)
        qout = qout.unsqueeze(-1)
        out = self.post(qout)
        return out

  print("Hybrid model created successfully!")

  model = HybridQNN()
  sample = torch.tensor([[0.3, 0.7]], dtype=torch.float32)
  out = model(sample)
  print("Hybrid model created successfully! Output example:", out.detach().numpy())

  optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
  criterion = nn.MSELoss()
  scheduler = StepLR(optimizer, step_size=30, gamma=0.1)

  X_train = torch.tensor(X_scaled, dtype=torch.float32)
  y_train = torch.tensor(2 * y - 1, dtype=torch.float32)

  dataset = torch.utils.data.TensorDataset(X_train, y_train)
  loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)

  n_epochs = 100
  for epoch in range(n_epochs):
    epoch_loss = 0.0
    for xb, yb in loader:
        optimizer.zero_grad()
        out = model(xb).squeeze(-1)
        loss = criterion(out, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item() * xb.shape[0]
    epoch_loss /= len(dataset)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1}/{n_epochs}, loss: {epoch_loss:.4f}")

  print("Hybrid model training complete!")
  ```
</details>
</div>